# KAIROS: NIFTY 50 Market Movement Analysis
UML501 — Machine Learning

Data acquisition, preprocessing, exploratory analysis, candidate predictors and classification targets.

**Execution:** Place the two source CSV files in the working directory or set `KAIROS_DATA_DIR` to their location. The optional acquisition function is not executed by default.

**Development assistance:** AI-assisted coding and editing were used in preparing this notebook; the methodology and results should be independently reviewed.

## 1. Problem statement
Investigate whether historical NIFTY 50 prices and India VIX can predict significant net price movements over the next five trading sessions. Forecasts are issued after market close. The provisional threshold is ±3%.

## 2. Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import os

## 3. Data collection
### 3.1 Source and collection method
Historical NIFTY 50 observations were obtained from NSE Indices. The optional acquisition function below reconstructs the original batched HTTP request procedure; the live endpoint has not been retested. Fixed CSV snapshots are used for reproducibility. The original India VIX acquisition script is unavailable.
### 3.2 Optional NIFTY acquisition code

In [ ]:
def collect_nifty_history(start_date, end_date):
    import requests
    import json
    import time
    from datetime import timedelta

    url = 'https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString'
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Content-Type': 'application/json',
        'X-Requested-With': 'XMLHttpRequest',
        'Referer': 'https://www.niftyindices.com/reports/historical-data',
        'Origin': 'https://www.niftyindices.com'
    }
    if start_date > end_date:
        raise ValueError('Start date must precede end date.')
    frames = []
    with requests.Session() as session:
        session.headers.update(headers)
        current_start = start_date
        while current_start <= end_date:
            current_end = min(current_start + timedelta(days=364), end_date)
            cinfo = ("{'name':'NIFTY 50','startDate':'" + current_start.strftime('%m/%d/%Y')
                     + "','endDate':'" + current_end.strftime('%m/%d/%Y')
                     + "','indexName':'NIFTY 50'}")
            for attempt in range(3):
                try:
                    response = session.post(url, json={'cinfo': cinfo}, timeout=90)
                    response.raise_for_status()
                    result = response.json()
                    if isinstance(result, dict):
                        result = result['d']
                    if isinstance(result, str):
                        result = json.loads(result)
                    if not isinstance(result, list):
                        raise ValueError('Unexpected response format')
                    if result:
                        frames.append(pd.DataFrame(result))
                    elif (current_end - current_start).days >= 7:
                        raise ValueError('Unexpected empty multi-day batch')
                    break
                except (requests.RequestException, ValueError, KeyError) as exc:
                    if attempt == 2:
                        raise RuntimeError(f'Failed batch {current_start} to {current_end}') from exc
                    time.sleep(2 ** attempt)
            current_start = current_end + timedelta(days=1)
            time.sleep(1.5)
    if not frames:
        raise ValueError('No observations returned')
    result = pd.concat(frames, ignore_index=True).rename(columns={
        'HistoricalDate': 'Date', 'date': 'Date', 'OPEN': 'Open',
        'HIGH': 'High', 'LOW': 'Low', 'CLOSE': 'Close'
    })[['Date', 'Open', 'High', 'Low', 'Close']]
    result['Date'] = pd.to_datetime(result['Date'], errors='raise')
    for col in ['Open', 'High', 'Low', 'Close']:
        result[col] = pd.to_numeric(result[col], errors='raise')
    return result.sort_values('Date').reset_index(drop=True)

# Optional historical request (not run):
# from datetime import date
# collected = collect_nifty_history(date(2006, 11, 1), date(2026, 11, 14))
# collected.to_csv('NIFTY_RECOLLECTED.csv', index=False)

### 3.3 Load source datasets
The source CSV files remain unchanged throughout the analysis.

In [ ]:
base = Path(os.environ.get('KAIROS_DATA_DIR', str(Path.cwd())))
nifty_file = base / 'KAIROS_NIFTY50_NSE_2006_2026.csv'
vix_file = base / 'KAIROS_INDIA_VIX_2008_2026.csv'
for file in (nifty_file, vix_file):
    if not file.is_file():
        raise FileNotFoundError(f'Upload {file.name} to {base} before continuing.')
nifty = pd.read_csv(nifty_file)
vix = pd.read_csv(vix_file)
print('NIFTY:', nifty.shape)
print('VIX:', vix.shape)
display(nifty.head(), vix.head())

## 4. Data preprocessing
### 4.1 Check column names and data types

In [ ]:
prices = ['Open', 'High', 'Low', 'Close']
assert set(nifty.columns) == set(['Date'] + prices)
assert set(vix.columns) == {'Date', 'VIX_Close'}
print(nifty.dtypes)
print(vix.dtypes)

### 4.2 Convert dates and sort chronologically

In [ ]:
for frame in (nifty, vix):
    frame['Date'] = pd.to_datetime(frame['Date'], errors='raise')
    frame.sort_values('Date', inplace=True)
    frame.reset_index(drop=True, inplace=True)
    assert frame.Date.notna().all()
    assert frame.Date.is_monotonic_increasing
print('NIFTY:', nifty.Date.min().date(), 'to', nifty.Date.max().date())
print('VIX:', vix.Date.min().date(), 'to', vix.Date.max().date())

### 4.3 Check missing values

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'missing values:')
    print(frame.isna().sum())
    assert frame.notna().all().all()

### 4.4 Check duplicate dates

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'duplicate dates:', frame.Date.duplicated().sum())
    assert frame.Date.is_unique

### 4.5 Validate OHLC and VIX values

In [ ]:
assert np.isfinite(nifty[prices]).all().all()
assert (nifty[prices] > 0).all().all()
assert (nifty.High >= nifty[['Open', 'Close', 'Low']].max(axis=1)).all()
assert (nifty.Low <= nifty[['Open', 'Close', 'High']].min(axis=1)).all()
assert np.isfinite(vix.VIX_Close).all() and (vix.VIX_Close > 0).all()
print('OHLC and VIX checks passed.')

### 4.6 Align the date ranges
Use the common observation period. Earlier NIFTY records are excluded from the joint experiment.

In [ ]:
start = max(nifty.Date.min(), vix.Date.min())
end = min(nifty.Date.max(), vix.Date.max())
assert start <= end
aligned = nifty.loc[nifty.Date.between(start, end)].copy()
vix_aligned = vix.loc[vix.Date.between(start, end)].copy()
assert vix_aligned.Date.isin(aligned.Date).all()
print('Common period:', start.date(), 'to', end.date())
print('Excluded NIFTY rows:', len(nifty) - len(aligned))
print('Aligned NIFTY rows:', len(aligned))
print('Aligned VIX rows:', len(vix_aligned))

### 4.7 Merge the datasets
Use a one-to-one left join to preserve aligned NIFTY sessions.

In [ ]:
df = aligned.merge(vix_aligned, on='Date', how='left', validate='one_to_one')
assert len(df) == len(aligned)
assert df.Date.is_unique and df.Date.is_monotonic_increasing
print('Merged shape before missing-value flag:', df.shape)
print('Missing VIX observations:', df.VIX_Close.isna().sum())
display(df.loc[df.VIX_Close.isna()])

### 4.8 Handle missing VIX observations
Flag missing readings before carrying forward the last available earlier value. A carried-forward reading is not an observed value.

In [ ]:
df['VIX_missing'] = df.VIX_Close.isna().astype(int)
print('Missing VIX dates:', df.loc[df.VIX_missing.eq(1), 'Date'].dt.date.tolist())
df['VIX_Close'] = df.VIX_Close.ffill()
assert df.notna().all().all()
assert len(df) == len(aligned)
print('Remaining missing values:', df.isna().sum().sum())
print('Cleaned dataset shape:', df.shape)
display(df.head())

### 4.9 Export the cleaned combined dataset
The market-data export excludes the internal imputation flag. A separate audit file records the affected dates and carried-forward VIX values. The internal flag is retained for feature-quality checks.

In [ ]:
output_path = base / 'KAIROS_COMBINED_CLEANED.csv'
audit_path = base / 'KAIROS_VIX_IMPUTATION_AUDIT.csv'
combined = df.drop(columns=['VIX_missing'])
combined.to_csv(output_path, index=False)
df.loc[df.VIX_missing.eq(1), ['Date', 'VIX_Close']].to_csv(audit_path, index=False)
print('Combined dataset:', combined.shape)
print('Imputation audit:', audit_path.name)
# from google.colab import files
# files.download(str(output_path))

## 5. Chronological experiment boundaries
Use 70% for training, 15% for validation and 15% for testing. EDA uses the training period only. The test partition is a procedural holdout rather than an independent external dataset because the historical snapshot was previously examined.

In [ ]:
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)
train = df.iloc[:train_end].copy()
for name, part in [('Train', train), ('Validation', df.iloc[train_end:val_end]), ('Test', df.iloc[val_end:])]:
    print(name, len(part), part.Date.min().date(), part.Date.max().date())

## 6. Exploratory Data Analysis (EDA)
### 6.1 Descriptive statistics

In [ ]:
display(train[prices + ['VIX_Close']].describe().round(2))

### 6.2 Market trends
NIFTY closing levels and implied volatility are plotted separately.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Close, linewidth=1)
axes[0].set(title='NIFTY 50 closing level', ylabel='Index points')
axes[1].plot(train.Date, train.VIX_Close, linewidth=1)
axes[1].set(title='India VIX', ylabel='Index level', xlabel='Date')
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### 6.3 Returns and rolling volatility
Compute daily percentage returns and trailing 20-session standard deviation.

In [ ]:
train['Return_1d'] = train.Close.pct_change() * 100
train['Volatility_20d'] = train.Return_1d.rolling(20).std()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Return_1d, linewidth=0.7)
axes[0].set(title='Daily NIFTY returns', ylabel='Return (%)')
axes[1].plot(train.Date, train.Volatility_20d, linewidth=1)
axes[1].set(title='20-session rolling volatility', ylabel='Daily std. dev. (%)', xlabel='Date')
fig.tight_layout()
plt.show()
print(train.Return_1d.describe().round(3))
print('Largest daily decline:', train.loc[train.Return_1d.idxmin(), ['Date','Return_1d']].to_dict())
print('Largest daily rise:', train.loc[train.Return_1d.idxmax(), ['Date','Return_1d']].to_dict())

### 6.4 VIX and market returns
Pearson correlation measures linear association, not predictive performance.

In [ ]:
train['VIX_Change'] = train.VIX_Close.pct_change() * 100
train['Abs_Return'] = train.Return_1d.abs()
cols = ['Return_1d', 'Abs_Return', 'VIX_Close', 'VIX_Change', 'Volatility_20d']
corr = train[cols].corr()
display(corr.round(3))
plt.figure(figsize=(7, 4))
plt.scatter(train.VIX_Change, train.Return_1d, s=8, alpha=0.35)
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.xlabel('Daily VIX change (%)')
plt.ylabel('Daily NIFTY return (%)')
plt.title('Same-session return and VIX change')
plt.tight_layout()
plt.show()

### 6.5 Candidate target distribution
Use a provisional five-session return threshold of ±3%. Exclude unknown outcomes.

In [ ]:
forward = (train.Close.shift(-5) / train.Close - 1) * 100
observed = forward.dropna()
labels = pd.Series(np.select([observed < -3, observed > 3], ['Major Down', 'Major Up'], default='Ordinary'), index=observed.index)
order = ['Major Down', 'Ordinary', 'Major Up']
counts = labels.value_counts().reindex(order, fill_value=0)
summary = pd.DataFrame({'Count': counts, 'Percent': counts / len(labels) * 100})
display(summary.round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(observed, bins=60, edgecolor='white')
for boundary in (-3, 3):
    axes[0].axvline(boundary, linestyle='--', linewidth=1.4)
axes[0].set(title='Five-session forward returns', xlabel='Return (%)', ylabel='Observations')
axes[1].bar(order, counts)
axes[1].set(title='Candidate target distribution', ylabel='Observations')
fig.tight_layout()
plt.show()
assert len(labels) == len(train) - 5
print('Excluded rows without a within-training outcome:', len(train) - len(labels))

### 6.6 EDA conclusions
Class imbalance requires macro F1 and per-class precision and recall alongside accuracy. Same-session correlations are not evidence of future forecasting skill.

## 7. Feature engineering
### 7.1 Candidate predictors
Forecasts are issued after market close. Trailing features may use earlier history across partition boundaries. The imputation flag is retained for data-quality checks only, not as a predictor. The first 20 sessions are warm-up rows.

In [ ]:
def make_features(data):
    features = pd.DataFrame(index=data.index)
    close = data['Close']
    vix_close = data['VIX_Close']
    daily_return = close.pct_change(fill_method=None) * 100
    features['Return_1d'] = daily_return
    features['Return_5d'] = close.pct_change(5, fill_method=None) * 100
    features['Volatility_20d'] = daily_return.rolling(20).std()
    features['MA_20_gap_pct'] = (close / close.rolling(20).mean() - 1) * 100
    features['Intraday_range_pct'] = (data['High'] - data['Low']) / data['Open'] * 100
    features['VIX_Close'] = vix_close
    features['VIX_change_1d'] = vix_close.pct_change(fill_method=None) * 100
    # Mark changes involving an imputed VIX as missing rather than inventing a move.
    unreliable = data['VIX_missing'].eq(1) | data['VIX_missing'].shift(1, fill_value=0).eq(1)
    features.loc[unreliable, 'VIX_change_1d'] = np.nan
    return features

X = make_features(df)
assert X.shape[1] == 7 and "VIX_missing" not in X.columns
X_train = X.iloc[:train_end].copy()
print('Candidate predictors:', X_train.shape[1])
print('Training rows with all predictors available:', X_train.dropna().shape[0])
display(X_train.iloc[20:25].round(3))

### 7.2 Feature validation and leakage checks
Prefix invariance tests for accidental use of future observations.

In [ ]:
for cutoff in (50, 250, train_end, val_end):
    pd.testing.assert_frame_equal(make_features(df.iloc[:cutoff]), X.iloc[:cutoff])
assert not np.isinf(X.to_numpy(dtype=float)).any()
assert X.index.equals(df.index)
print('Prefix-invariance, finite-value and alignment checks passed.')
print('Training missing counts:')
print(X_train.isna().sum())

## 8. Target variable creation
### 8.1 Five-session classification labels
Returns above +3% map to 1, below −3% to −1, and the inclusive interval to 0. A numerical tolerance protects threshold boundaries. Outcomes are generated separately in each partition.

In [ ]:
"""Five-session classification labels, with partition boundary protection."""
import numpy as np
import pandas as pd


def make_targets(data, horizon=5, threshold=3.0):
    if horizon < 1 or threshold <= 0:
        raise ValueError('horizon and threshold must be positive')
    if not data.Date.is_monotonic_increasing or not data.Date.is_unique:
        raise ValueError('Dates must be unique and sorted')
    forward = (data.Close.shift(-horizon) / data.Close - 1) * 100
    boundary = np.isclose(forward.abs(), threshold, rtol=0, atol=1e-10)
    target = pd.Series(pd.NA, index=data.index, dtype='Int64')
    target.loc[forward.notna()] = 0
    target.loc[(forward > threshold) & ~boundary] = 1
    target.loc[(forward < -threshold) & ~boundary] = -1
    return pd.DataFrame({'Outcome_date': data.Date.shift(-horizon),
                         'Forward_return_5d_pct': forward, 'Target': target})


### 8.2 Prepare aligned training observations
Forward returns and outcome dates are retained for evaluation only. Exclude unavailable features and unknown outcomes.

In [ ]:
partitions = {'train': slice(0, train_end), 'validation': slice(train_end, val_end), 'test': slice(val_end, len(df))}
feature_columns = X.columns.tolist()
training_labels = make_targets(df.iloc[partitions['train']])
training_mask = X.iloc[partitions['train']].notna().all(axis=1) & training_labels.Target.notna()
X_fit = X.iloc[partitions['train']].loc[training_mask, feature_columns].copy()
y_fit = training_labels.loc[training_mask, 'Target'].astype(int)
print('Eligible training observations:', len(y_fit))
print('Excluded for unknown outcomes:', training_labels.Target.isna().sum())
print('Excluded for unavailable predictors among labelled rows:', ((~X.iloc[partitions['train']].notna().all(axis=1)) & training_labels.Target.notna()).sum())
counts_fit = y_fit.value_counts().reindex([-1, 0, 1], fill_value=0)
print(pd.DataFrame({'Count': counts_fit, 'Percent': (counts_fit / len(y_fit) * 100).round(2)}))

### 8.3 Validate target boundaries and alignment

In [ ]:
for future, expected in [(96, -1), (97, 0), (100, 0), (103, 0), (104, 1)]:
    example = pd.DataFrame({'Date': pd.date_range('2020-01-01', periods=6), 'Close': [100]*5 + [future]})
    check = make_targets(example)
    assert check.Target.iloc[0] == expected
    assert check.Target.iloc[1:].isna().all()
for part in partitions.values():
    block = df.iloc[part]
    result = make_targets(block)
    assert result.Target.tail(5).isna().all()
    assert result.Outcome_date.dropna().le(block.Date.max()).all()
    assert result.Target.notna().sum() == len(block) - 5
assert X_fit.index.equals(y_fit.index)
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(X_fit.columns)
assert X_fit.notna().all().all()
print('Target threshold, partition boundary and feature alignment checks passed.')

## 9. Final dataset preparation
### 9.1 Training feature review
Review training-period missingness, variability and pairwise correlations. Correlation screening is descriptive; feature selection is deferred to chronological model validation.

In [ ]:
# Review training features only; do not use validation or test correlations.
review = pd.DataFrame({'Missing': X_train.isna().sum(),
                       'Unique_values': X_train.nunique(),
                       'Std_dev': X_train.std()})
display(review.round(4))
assert X_train.nunique().gt(1).all()
correlations = X_train.corr()
pairs = correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1)).stack()
strong_pairs = pairs[pairs.abs() >= 0.80].sort_values(key=abs, ascending=False)
print('Feature correlations with |r| >= 0.80:')
print(strong_pairs.round(3) if len(strong_pairs) else 'None')

### 9.2 Assemble partition-safe datasets
Only observations with all seven features and a known outcome within their chronological partition are eligible. The research table contains market values, features, outcome metadata and partition identifiers. Only the explicit feature allowlist is used as model input.

In [ ]:
"""Assemble partition-safe research datasets from aligned prices and features."""
import pandas as pd
import numpy as np

FEATURE_COLUMNS = ['Return_1d', 'Return_5d', 'Volatility_20d', 'MA_20_gap_pct',
                   'Intraday_range_pct', 'VIX_Close', 'VIX_change_1d']
MARKET_COLUMNS = ['Date', 'Open', 'High', 'Low', 'Close', 'VIX_Close']


def build_datasets(df, X, partitions, make_targets):
    assert df.index.is_unique and X.index.equals(df.index)
    assert df.Date.is_unique and df.Date.is_monotonic_increasing
    assert X.columns.tolist() == FEATURE_COLUMNS
    pd.testing.assert_series_equal(X.VIX_Close, df.VIX_Close)
    assert not np.isinf(X.to_numpy(dtype=float)).any()
    assert list(partitions) == ['train', 'validation', 'test']
    used = []
    datasets = {}
    audit = []
    for name, section in partitions.items():
        market = df.iloc[section]
        features = X.iloc[section]
        used.extend(market.index.tolist())
        labels = make_targets(market)
        assert labels.index.equals(market.index)
        feature_ok = features.notna().all(axis=1)
        target_ok = labels.Target.notna()
        eligible = feature_ok & target_ok
        # VIX_Close is already present among the market columns.
        joined = market[MARKET_COLUMNS].join(features.drop(columns='VIX_Close')).join(labels)
        joined['Partition'] = name
        part = joined.loc[eligible].copy()
        part['Target'] = part.Target.astype(int)
        assert not part.empty and part.notna().all().all()
        assert part.Target.isin([-1, 0, 1]).all()
        assert part.Outcome_date.gt(part.Date).all()
        assert part.Outcome_date.le(market.Date.max()).all()
        assert part.Date.is_unique and part.Date.is_monotonic_increasing
        datasets[name] = part
        audit.append({'Partition': name, 'Original': len(market),
                      'Unavailable_target': int((~target_ok).sum()),
                      'Unavailable_features_with_target': int((target_ok & ~feature_ok).sum()),
                      'Eligible': len(part)})
    assert used == df.index.tolist()
    final = pd.concat(datasets.values(), ignore_index=True)
    assert final.Date.is_unique and final.Date.is_monotonic_increasing
    assert len(final) == sum(len(part) for part in datasets.values())
    for left, right in [('train', 'validation'), ('validation', 'test')]:
        assert datasets[left].Outcome_date.max() < datasets[right].Date.min()
    # Prediction inputs: latest five sessions with complete features but unknown outcomes.
    recent = df.tail(5)
    recent_features = X.loc[recent.index]
    prediction = recent[['Date']].join(recent_features).dropna().copy()
    assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(prediction.columns)
    assert set(prediction.Date).isdisjoint(final.Date)
    return final, datasets, prediction, pd.DataFrame(audit)

### 9.3 Validate and summarize the eligible observations
Record exclusions separately for unavailable outcomes and unavailable features. Retain the latest five sessions with complete predictors separately for inference; no future labels are attached.

In [ ]:
final_dataset, datasets, prediction_inputs, dataset_audit = build_datasets(df, X, partitions, make_targets)
print('Dataset preparation audit:')
display(dataset_audit)
print('Final research dataset:', final_dataset.shape)
print('Prediction inputs:', prediction_inputs.shape)

# The model input allowlist is independent of the research CSV schema.
X_fit = datasets['train'][FEATURE_COLUMNS].copy()
y_fit = datasets['train']['Target'].copy()
assert X_fit.index.equals(y_fit.index)
assert len(X_fit) == len(training_labels.loc[training_mask])
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct', 'Partition'}.intersection(X_fit.columns)
print('Training input shape:', X_fit.shape)
print('Training target shape:', y_fit.shape)

### 9.4 Export reproducible datasets
Export the labelled research table, chronological subsets, inference inputs and an exclusion audit. Source snapshots are not overwritten. Validation and test outcomes are not summarized during preparation.

In [ ]:
export_dir = Path(os.environ.get('KAIROS_OUTPUT_DIR', str(base / 'kairos_datasets')))
export_dir.mkdir(parents=True, exist_ok=True)
exports = {'KAIROS_FINAL_DATASET.csv': final_dataset,
           'KAIROS_TRAIN.csv': datasets['train'],
           'KAIROS_VALIDATION.csv': datasets['validation'],
           'KAIROS_TEST.csv': datasets['test'],
           'KAIROS_PREDICTION_INPUTS.csv': prediction_inputs,
           'KAIROS_DATASET_AUDIT.csv': dataset_audit}
for filename, frame in exports.items():
    path = export_dir / filename
    frame.to_csv(path, index=False)
    saved = pd.read_csv(path)
    assert saved.shape == frame.shape
    assert saved.columns.tolist() == frame.columns.tolist()
    print(filename, saved.shape)

## 10. Baseline models
### 10.1 Benchmark definitions and evaluation metrics
A majority-class classifier and a trailing five-session momentum rule establish reference performance. Both use information available by the prediction date. Accuracy, balanced accuracy, macro F1 and per-class metrics are reported on the validation partition only.

In [ ]:
"""Validation benchmarks for five-session market movement classification."""
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

CLASSES = [-1, 0, 1]
CLASS_NAMES = ['Major Down', 'Ordinary', 'Major Up']


def momentum_predict(X, threshold=3.0):
    """Classify trailing five-session return using the target thresholds."""
    if threshold <= 0:
        raise ValueError('Threshold must be positive')
    returns = X['Return_5d'].to_numpy(dtype=float)
    if not np.isfinite(returns).all():
        raise ValueError('Momentum inputs must be finite')
    boundary = np.isclose(np.abs(returns), threshold, rtol=0, atol=1e-10)
    return np.select([(returns < -threshold) & ~boundary,
                      (returns > threshold) & ~boundary], [-1, 1], default=0)


def evaluate_predictions(y_true, y_pred):
    """Score every target class, including classes never predicted."""
    report = classification_report(y_true, y_pred, labels=CLASSES,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    summary = {'Accuracy': accuracy_score(y_true, y_pred),
               'Balanced accuracy': balanced_accuracy_score(y_true, y_pred),
               'Macro F1': f1_score(y_true, y_pred, labels=CLASSES,
                                    average='macro', zero_division=0),
               'Major Down recall': report['Major Down']['recall'],
               'Major Up recall': report['Major Up']['recall']}
    per_class = pd.DataFrame(report).T.loc[CLASS_NAMES,
                                         ['precision', 'recall', 'f1-score', 'support']]
    matrix = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES),
                          index=CLASS_NAMES, columns=CLASS_NAMES)
    return summary, per_class, matrix

### 10.2 Fit the majority classifier and generate validation predictions
The majority class is learned from training labels. The momentum rule uses the same fixed ±3% thresholds as the target and does not estimate parameters. Neither uses the validation outcomes to make predictions.

In [ ]:
X_val = datasets['validation'][FEATURE_COLUMNS].copy()
y_val = datasets['validation']['Target'].copy()
assert X_fit.columns.equals(X_val.columns)
assert X_val.index.equals(y_val.index)
assert X_fit.index.intersection(X_val.index).empty
assert set(y_fit.unique()) == set(CLASSES)
assert set(y_val.unique()) == set(CLASSES)

majority = DummyClassifier(strategy='most_frequent')
majority.fit(X_fit, y_fit)
predictions = {'Majority class': majority.predict(X_val),
               'Momentum': momentum_predict(X_val)}
print('Training class counts:')
print(y_fit.value_counts().sort_index())
print('Validation class counts:')
print(y_val.value_counts().sort_index())
assert np.all(predictions['Majority class'] == y_fit.mode().iloc[0])

### 10.3 Validation performance
Confusion matrices use the fixed class order Major Down, Ordinary, Major Up. Precision and recall for classes with no predicted observations are reported as zero where undefined. The test partition is reserved for final model evaluation.

In [ ]:
baseline_rows = []
for name, pred in predictions.items():
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    baseline_rows.append({'Model': name, **metrics})
    print(name, '- validation class metrics')
    display(per_class.round(3))
    print('Confusion matrix (rows: actual; columns: predicted)')
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
baseline_results = pd.DataFrame(baseline_rows).set_index('Model')
display(baseline_results.round(4))

## 11. Multiclass Logistic Regression
### 11.1 Pipeline definition
Standardize predictors using training statistics, then fit a regularized multinomial logistic classifier. Compare unweighted and inverse-frequency class weighting using fixed initial settings (C=1).

In [ ]:
"""Multiclass logistic regression candidates for KAIROS."""
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def build_logistic(class_weight=None):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(solver='lbfgs', C=1.0,
                                          max_iter=2000, class_weight=class_weight))
    ])

### 11.2 Training and validation
Fit each pipeline using training observations only. Evaluate on the same validation observations as the baselines. Check convergence, class ordering and predicted probabilities.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

logistic_models = {
    'Logistic (unweighted)': build_logistic(),
    'Logistic (balanced)': build_logistic(class_weight='balanced')
}
logistic_predictions = {}
logistic_probabilities = {}
logistic_rows = []
for name, model in logistic_models.items():
    with warnings.catch_warnings():
        warnings.simplefilter('error', ConvergenceWarning)
        model.fit(X_fit, y_fit)
    # Standardization statistics must come only from the training data.
    np.testing.assert_allclose(model.named_steps['scaler'].mean_, X_fit.mean().to_numpy())
    assert model.classes_.tolist() == CLASSES
    probabilities = model.predict_proba(X_val)
    pred = model.predict(X_val)
    assert probabilities.shape == (len(X_val), len(CLASSES))
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.array_equal(pred, model.classes_[probabilities.argmax(axis=1)])
    logistic_predictions[name] = pred
    logistic_probabilities[name] = probabilities
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    logistic_rows.append({'Model': name, **metrics})
    print(name)
    display(per_class.round(3))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
logistic_results = pd.DataFrame(logistic_rows).set_index('Model')
comparison = pd.concat([baseline_results, logistic_results])
display(comparison.round(4))

### 11.3 Class weights and probability example
Report inverse-frequency training weights and an illustrative validation observation. No final model selection or test evaluation is performed.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(class_weight='balanced', classes=np.array(CLASSES), y=y_fit.to_numpy())
display(pd.DataFrame({'Class': CLASS_NAMES, 'Training_count': y_fit.value_counts().reindex(CLASSES).to_numpy(),
                      'Balanced_weight': weights}).round(4))
# Show a validation forecast with its corresponding observed outcome.
example = pd.DataFrame({name: probabilities[0] for name, probabilities in logistic_probabilities.items()}, index=CLASS_NAMES)
print('Validation example date:', datasets['validation'].Date.iloc[0].date())
print('Observed label:', y_val.iloc[0])
display(example.round(4))

## 12. Decision trees
### 12.1 Model definition
Gini-based classification trees are fitted without feature scaling. Two depth-limited candidates (maximum depth 4, minimum leaf size 30) compare ordinary and balanced class weighting. An unrestricted unweighted tree provides an overfitting diagnostic.

In [ ]:
"""Decision tree candidates for market movement classification."""
from sklearn.tree import DecisionTreeClassifier


def build_tree(class_weight=None, max_depth=4, min_samples_leaf=30):
    return DecisionTreeClassifier(criterion='gini', max_depth=max_depth,
                                  min_samples_leaf=min_samples_leaf,
                                  class_weight=class_weight, random_state=42)

### 12.2 Training and validation
Fit on training observations only and evaluate on the same validation partition as previous models. Record training macro F1, validation metrics and tree complexity. The unrestricted tree is diagnostic rather than a tuned candidate.

In [ ]:
tree_models = {
    'Tree (unweighted)': build_tree(),
    'Tree (balanced)': build_tree(class_weight='balanced'),
    'Tree (unrestricted)': build_tree(max_depth=None, min_samples_leaf=1)
}
tree_predictions = {}
tree_rows = []
for name, model in tree_models.items():
    model.fit(X_fit, y_fit)
    pred = model.predict(X_val)
    probabilities = model.predict_proba(X_val)
    assert model.classes_.tolist() == CLASSES
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    assert np.array_equal(pred, model.classes_[probabilities.argmax(axis=1)])
    tree_predictions[name] = pred
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    train_metrics, _, _ = evaluate_predictions(y_fit, model.predict(X_fit))
    tree_rows.append({'Model': name, **metrics,
                      'Train macro F1': train_metrics['Macro F1'],
                      'Depth': model.get_depth(), 'Leaves': model.get_n_leaves()})
    print(name)
    display(per_class.round(3))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
tree_results = pd.DataFrame(tree_rows).set_index('Model')
display(tree_results.round(4))
comparison = pd.concat([baseline_results, logistic_results, tree_results[baseline_results.columns]])
display(comparison.round(4))

### 12.3 Inspect the learned tree
Report the balanced tree root split and complexity, and plot the first three levels. Weighted class frequencies affect split impurity and predicted class scores.

In [ ]:
from sklearn.tree import plot_tree

balanced_tree = tree_models['Tree (balanced)']
root_feature = FEATURE_COLUMNS[balanced_tree.tree_.feature[0]]
root_threshold = balanced_tree.tree_.threshold[0]
print('Balanced tree root:', root_feature, '<=', round(root_threshold, 4))
print('Root Gini impurity:', round(balanced_tree.tree_.impurity[0], 4))
print('Actual depth:', balanced_tree.get_depth())
print('Leaf count:', balanced_tree.get_n_leaves())
assert balanced_tree.get_depth() <= 4
assert balanced_tree.tree_.n_node_samples[balanced_tree.tree_.children_left == -1].min() >= 30
plt.figure(figsize=(15, 5))
plot_tree(balanced_tree, feature_names=FEATURE_COLUMNS, class_names=CLASS_NAMES,
          max_depth=2, filled=True, rounded=False, fontsize=9)
plt.title('Balanced decision tree (first three levels)')
plt.tight_layout()
plt.show()

## 13. Random forest
### 13.1 Model definition
Compare two forests of 200 bootstrap trees with fixed initial depth, minimum leaf size and square-root feature sampling. Only class weighting differs.

In [ ]:
"""Random forest candidates for market movement classification."""
from sklearn.ensemble import RandomForestClassifier


def build_forest(class_weight=None, n_estimators=200, max_depth=6, min_samples_leaf=20):
    return RandomForestClassifier(n_estimators=n_estimators, criterion='gini',
                                  max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                                  max_features='sqrt', bootstrap=True,
                                  class_weight=class_weight, random_state=42, n_jobs=1)

### 13.2 Training and validation
Fit both forests on the training observations and compare with previous candidates on the same validation partition. Record training macro F1 as a complexity diagnostic.

In [ ]:
forest_models = {
    'Forest (unweighted)': build_forest(),
    'Forest (balanced)': build_forest(class_weight='balanced')
}
forest_predictions = {}
forest_rows = []
for name, model in forest_models.items():
    model.fit(X_fit, y_fit)
    pred = model.predict(X_val)
    probabilities = model.predict_proba(X_val)
    assert model.classes_.tolist() == CLASSES
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    assert np.array_equal(pred, model.classes_[probabilities.argmax(axis=1)])
    assert len(model.estimators_) == model.n_estimators
    forest_predictions[name] = pred
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    train_metrics, _, _ = evaluate_predictions(y_fit, model.predict(X_fit))
    forest_rows.append({'Model': name, **metrics,
                        'Train macro F1': train_metrics['Macro F1']})
    print(name)
    display(per_class.round(3))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
forest_results = pd.DataFrame(forest_rows).set_index('Model')
display(forest_results.round(4))
comparison = pd.concat([baseline_results, logistic_results, tree_results[baseline_results.columns],
                        forest_results[baseline_results.columns]])
display(comparison.round(4))

### 13.3 Probability aggregation and feature importance
Inspect tree diversity and verify probability averaging. Impurity-based importance is descriptive and can be biased by correlated predictors and split opportunities.

In [ ]:
balanced_forest = forest_models['Forest (balanced)']
assert all(tree.get_depth() <= 6 for tree in balanced_forest.estimators_)
assert all(tree.max_features_ == 2 for tree in balanced_forest.estimators_)
print('Number of trees:', len(balanced_forest.estimators_))
print('Candidate features per split:', balanced_forest.estimators_[0].max_features_)
print('Mean tree depth:', np.mean([tree.get_depth() for tree in balanced_forest.estimators_]))
# RandomForest averages individual tree probability vectors.
example = X_val.iloc[[0]]
individual = np.vstack([tree.predict_proba(example.to_numpy())[0]
                        for tree in balanced_forest.estimators_])
np.testing.assert_allclose(individual.mean(axis=0), balanced_forest.predict_proba(example)[0])
print('First validation observation:', datasets['validation'].Date.iloc[0].date())
display(pd.DataFrame(individual[:5], columns=CLASS_NAMES).round(4))
print('Mean across all trees:', individual.mean(axis=0).round(4))
# Impurity-based importance is descriptive, not causal or out-of-sample evidence.
importance = pd.Series(balanced_forest.feature_importances_, index=FEATURE_COLUMNS).sort_values()
display(importance.sort_values(ascending=False).round(4))
importance.plot.barh(figsize=(8, 4), title='Balanced forest: impurity-based feature importance')
plt.xlabel('Normalized impurity reduction')
plt.tight_layout()
plt.show()

## 14. Gradient boosting
### 14.1 Model definition
Compare multiclass gradient boosting candidates with fixed initial settings: 100 stages, learning rate 0.05, depth 2 and minimum leaf size 30. Balanced sample weights are calculated solely from the training labels.

In [ ]:
"""Gradient boosting candidates for KAIROS."""
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight


def build_boosting(n_estimators=100, learning_rate=0.05, max_depth=2, min_samples_leaf=30):
    return GradientBoostingClassifier(loss='log_loss', n_estimators=n_estimators,
                                      learning_rate=learning_rate, max_depth=max_depth,
                                      min_samples_leaf=min_samples_leaf, subsample=1.0,
                                      random_state=42)


def fit_boosting(model, X, y, balanced=False):
    weights = compute_sample_weight('balanced', y) if balanced else None
    return model.fit(X, y, sample_weight=weights)

### 14.2 Training and validation
Train both candidates on the training partition. Compare validation performance against the same baselines and earlier models. Record training macro F1 for diagnostic purposes.

In [ ]:
boost_models = {'Boosting (unweighted)': build_boosting(),
                'Boosting (balanced)': build_boosting()}
boost_predictions = {}
boost_rows = []
for name, model in boost_models.items():
    fit_boosting(model, X_fit, y_fit, balanced=(name == 'Boosting (balanced)'))
    pred = model.predict(X_val)
    proba = model.predict_proba(X_val)
    assert model.classes_.tolist() == CLASSES
    assert model.estimators_.shape == (100, 3)
    np.testing.assert_allclose(proba.sum(axis=1), 1.0)
    assert np.array_equal(pred, model.classes_[proba.argmax(axis=1)])
    boost_predictions[name] = pred
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    train_metrics, _, _ = evaluate_predictions(y_fit, model.predict(X_fit))
    boost_rows.append({'Model': name, **metrics, 'Train macro F1': train_metrics['Macro F1']})
    print(name)
    display(per_class.round(4))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
boost_results = pd.DataFrame(boost_rows).set_index('Model')
display(boost_results.round(4))
comparison = pd.concat([baseline_results, logistic_results, tree_results[baseline_results.columns],
                        forest_results[baseline_results.columns], boost_results[baseline_results.columns]])
print(comparison.sort_values('Macro F1', ascending=False).round(4).to_string())
print(boost_results.round(4).to_string())

### 14.3 Stage diagnostics
Inspect training and validation macro F1 as the ensemble grows. All 100 stages are retained for the initial comparison; stage count is not optimized here.

In [ ]:
from sklearn.metrics import f1_score

balanced_boost = boost_models['Boosting (balanced)']
# Diagnostic only; no validation-based early stopping or stage selection.
stages = np.arange(1, balanced_boost.n_estimators_ + 1)
train_curve = [f1_score(y_fit, pred, labels=CLASSES, average='macro', zero_division=0)
               for pred in balanced_boost.staged_predict(X_fit)]
val_curve = [f1_score(y_val, pred, labels=CLASSES, average='macro', zero_division=0)
             for pred in balanced_boost.staged_predict(X_val)]
np.testing.assert_allclose(val_curve[-1], boost_results.loc['Boosting (balanced)', 'Macro F1'])
assert all(tree.get_depth() <= 2 for tree in balanced_boost.estimators_.ravel())
weights = compute_sample_weight('balanced', y_fit)
print('Total training weight by class:')
print(pd.Series(weights, index=y_fit.index).groupby(y_fit).sum())
print('Stages:', balanced_boost.n_estimators_, '| Trees:', balanced_boost.estimators_.size)
print('Stage diagnostics:')
print(pd.DataFrame({'Stage': stages, 'Training macro F1': train_curve,
                    'Validation macro F1': val_curve}).iloc[[0, 9, 24, 49, 74, 99]].round(4).to_string(index=False))
plt.figure(figsize=(8, 4))
plt.plot(stages, train_curve, label='Training')
plt.plot(stages, val_curve, label='Validation')
plt.xlabel('Boosting stages')
plt.ylabel('Macro F1')
plt.title('Balanced gradient boosting: learning stages')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Hyperparameter tuning
### 15.1 Search definition
Use four expanding-window folds within the training partition, with 400 scored observations per fold and a five-row gap. Confirm that every training outcome precedes the first fold evaluation date. Select one configuration per family by mean fold macro F1. Class weights and scaling are fitted separately in each fold.

In [ ]:
"""Chronological hyperparameter search with fold-local fitting and weighting."""
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.metrics import f1_score

TUNING_GRIDS = {
    'Logistic': {'C': [0.01, 0.1, 1.0, 10.0], 'class_weight': [None, 'balanced']},
    'Tree': {'max_depth': [3, 5, 7], 'min_samples_leaf': [20, 50], 'class_weight': [None, 'balanced']},
    'Forest': {'max_depth': [4, 8], 'min_samples_leaf': [10, 30], 'class_weight': [None, 'balanced']},
    'Boosting': {'n_estimators': [50, 100], 'max_depth': [1, 2], 'balanced': [False, True]}
}


def chronological_folds(training, n_splits=4, test_size=400, gap=5):
    dates = pd.to_datetime(training['Date'])
    outcomes = pd.to_datetime(training['Outcome_date'])
    assert dates.is_unique and dates.is_monotonic_increasing
    splitter = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap)
    folds = list(splitter.split(training))
    for tr, va in folds:
        assert tr[-1] + gap < va[0]
        assert outcomes.iloc[tr].max() < dates.iloc[va].min()
        assert set(training.Target.iloc[tr].unique()) == {-1, 0, 1}
        assert set(training.Target.iloc[va].unique()) == {-1, 0, 1}
    return folds


def search_family(name, X, y, folds, fit_candidate):
    rows = []
    for params in ParameterGrid(TUNING_GRIDS[name]):
        scores = []
        for tr, va in folds:
            model = fit_candidate(name, params, X.iloc[tr], y.iloc[tr])
            pred = model.predict(X.iloc[va])
            scores.append(f1_score(y.iloc[va], pred, labels=[-1, 0, 1],
                                   average='macro', zero_division=0))
        rows.append({'Family': name, 'Parameters': params,
                     **{f'Fold {i+1}': score for i, score in enumerate(scores)},
                     'Mean CV macro F1': np.mean(scores), 'SD CV macro F1': np.std(scores)})
    return pd.DataFrame(rows).sort_values('Mean CV macro F1', ascending=False, kind='stable').reset_index(drop=True)

### 15.2 Chronological search
Search a finite grid of initial configurations. For forest and boosting, non-grid settings remain fixed at the previously defined values. Report the top three configurations per family and fold-level variability.

In [ ]:
def fit_candidate(name, params, X, y):
    if name == 'Logistic':
        model = build_logistic(class_weight=params['class_weight'])
        model.set_params(classifier__C=params['C'])
    elif name == 'Tree':
        model = build_tree(**params)
    elif name == 'Forest':
        model = build_forest(**params)
    elif name == 'Boosting':
        settings = {k: v for k, v in params.items() if k != 'balanced'}
        return fit_boosting(build_boosting(**settings), X, y, balanced=params['balanced'])
    else:
        raise ValueError(name)
    return model.fit(X, y)


training = datasets['train']
assert training.index.equals(X_fit.index) and X_fit.index.equals(y_fit.index)
folds = chronological_folds(training)
fold_audit = pd.DataFrame([{'Fold': i+1, 'Train rows': len(tr), 'Gap rows': va[0]-tr[-1]-1,
                            'CV rows': len(va), 'Latest training outcome': training.Outcome_date.iloc[tr].max(),
                            'First CV date': training.Date.iloc[va].min()}
                           for i, (tr, va) in enumerate(folds)])
display(fold_audit)
search_tables = {}
for name in TUNING_GRIDS:
    search_tables[name] = search_family(name, X_fit, y_fit, folds, fit_candidate)
    print(name, '| configurations:', len(search_tables[name]))
    display(search_tables[name].head(3).round(4))
search_results = pd.concat(search_tables.values(), ignore_index=True)
print('Configurations:', len(search_results), '| CV fits:', len(search_results)*len(folds))

### 15.3 Validation comparison
Refit each family winner on all eligible training observations, then assess the same external validation period used for initial comparisons. No test predictions or final model selection are performed.

In [ ]:
tuned_models = {}
tuned_rows = []
for name, results in search_tables.items():
    best = results.iloc[0]
    model = fit_candidate(name, best['Parameters'], X_fit, y_fit)
    tuned_models[name] = model
    metrics, per_class, matrix = evaluate_predictions(y_val, model.predict(X_val))
    tuned_rows.append({'Model': name + ' (tuned)', **metrics,
                       'Mean CV macro F1': best['Mean CV macro F1'],
                       'SD CV macro F1': best['SD CV macro F1'],
                       'Parameters': best['Parameters']})
    assert matrix.to_numpy().sum() == len(y_val)
    print(name, '| selected configuration:', best['Parameters'])
    display(matrix)
tuned_results = pd.DataFrame(tuned_rows).set_index('Model')
display(tuned_results.round(4))
print(tuned_results.round(4).to_string())
comparison = pd.concat([comparison[baseline_results.columns], tuned_results[baseline_results.columns]])
print(comparison.sort_values('Macro F1', ascending=False).round(4).to_string())
# Export diagnostics only; final model selection and test scoring are separate stages.
search_results.to_csv(export_dir / 'KAIROS_CV_SEARCH.csv', index=False)

## 16. Scope and limitations
The feature set and target definition remain provisional. The validation period has been used repeatedly for exploratory comparisons. Grid search uses training-only chronological folds with a five-session gap and an explicit outcome-date check. The search is limited to the specified configurations; final model selection and test scoring remain pending. Fixed historical snapshots are used; live acquisition has not been retested.